In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from tqdm import tqdm


In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving relations_cancer.xlsx to relations_cancer.xlsx


In [ ]:
df= pd.read_csv('ner-heart.csv')

In [ ]:
df.shape

(33296, 12)

In [ ]:
df = df.drop_duplicates(
    subset=["sentence"]
).reset_index(drop=True)

print("After removing duplicates:", df.shape)


After removing duplicates: (33294, 12)


In [ ]:
df.columns

Index(['PMID', 'authors', 'affiliation', 'country', 'journal', 'year', 'title',
       'abstract', 'filtered_abstract', 'sentence', 'tokens', 'tags'],
      dtype='object')

In [ ]:
df.shape

(33294, 12)

In [ ]:
df.head()

,PMID,authors,affiliation,country,journal,year,title,abstract,filtered_abstract,sentence,tokens,tags
0,41224089,Zeng Miaolin; Zhai Xiangyang; Zhang Man; Du Ha...,"School of Chinese Medicine, Wenzhou Medical Un...",China,Journal of ethnopharmacology,2026,Therapeutic effects of Danhong injection on is...,"Danhong injection (DI), a standardized traditi...",has been validated in clinical studies for its...,has been validated in clinical studies for its...,"['has', 'been', 'validated', 'in', 'clinical',...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,41183695,Wei Pengcheng; Li Duan; Chen Wenxuan; Zhang Xi...,"School of Forensic Medicine, Xinxiang Medical ...",China,Cellular signalling,2026,Targeting GALNT4 suppresses atherosclerosis vi...,Endothelial inflammation is a critical driver ...,although nacetylgalactosaminyltransferase 4 ga...,although nacetylgalactosaminyltransferase 4 ga...,"['although', 'nacetylgalactosaminyltransferase...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
2,41045882,Hao Jinkui; Shah Nilay S; Zhou Bo,"Department of Radiology, Northwestern Universi...",Usa,Medical image analysis,2026,S<sup>2</sup>CAC: Semi-supervised coronary art...,Coronary artery calcium (CAC) scoring plays a ...,coronary artery calcium cac scoring plays a pi...,coronary artery calcium cac scoring plays a pi...,"['coronary', 'artery', 'calcium', 'cac', 'scor...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
3,40581083,Khnanisho Michael E; Holle Alejandro M; Tummal...,"Albany Medical College, Albany, NY, USA; Depar...",Usa,Journal of shoulder and elbow surgery,2026,Chewing tobacco use is associated with increas...,Arthroscopic rotator cuff repair (RCR) is a co...,0.7 or 6.84 myocardial infarction 1.1 vs 0.9 ...,0.7 or 6.84 myocardial infarction 1.1 vs 0.9 ...,"['0.7', 'or', '6.84', 'myocardial', 'infarctio...","['O', 'O', 'O', 'B-heart_disease', 'I-heart_di..."
4,41120039,He William J; Prescott Brenton R; Xanthakis Va...,"Framingham Heart Study, National Heart, Lung, ...",Usa,American heart journal,2026,Association of obesity subphenotypes with indi...,Previous studies have reported that obesity-re...,previous studies have reported that obesityrel...,previous studies have reported that obesityrel...,"['previous', 'studies', 'have', 'reported', 't...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


In [ ]:
POSITIVE = [
"constitute", "major contributor", "major source", "primary contributor", "dominant source",
"significant proportion", "central role", "substantial source", "leading contributor", "key driver",
"major determinant", "principal source", "main driver", "primary source", "fundamental",
"large fraction", "chiefly responsible", "predominant factor", "major share", "instrumental",
"critical component", "decisive factor", "cornerstone", "largely attributable", "most significant contributor",
"dominant determinant", "strongly predictive", "causally linked", "positive correlation", "lead to", "persistent","risk factor",
"contribute significantly", "important contributor", "important role", "key factor", "substantial impact",
"important source", "notable contributor", "important pathway", "exposure pathways", "relevant source",
"significant contribution", "considerable source", "strong influence", "material contributor",
"prominent factor", "salient contributor", "meaningful source", "non-negligible contributor",
"consistent predictor", "contribute materially", "established factor", "recognized source",
"non-trivial source", "independently associated", "significantly associated","considerable risk",
"associated", "linked", "correlated", "connected", "related", "association", "relationship",
"implicated", "involved", "contribute processes", "co-vary", "connection", "linkage", "component",
"facet", "part", "characteristic", "feature", "parameter", "variable", "evidence",
"may contribute", "may play role", "may represent source", "suggested contributor",
"thought to contribute", "appear influence", "could act source", "potentially involved",
"may account", "suspected contributors", "might contribute", "potentially source",
"hypothesized contribute", "proposed contributor", "possible source", "putative contributor",
"candidate", "plausibly linked", "conjectured involved", "tentatively associated", "could explain",
"not ruled out", "suggestive evidence", "borderline association", "trend", "directionally consistent", " attributable to","leading cause",
"increasingly recognized", "growing attention", "emerging important factor", "underexplored",
"overlooked source", "insufficiently studied", "neglected pathway", "understudied source",
"limited attention", "poorly characterized", "nascent source", "hidden contributor",
"unrecognized pathway", "blind spot", "non-conventional source", "non-traditional contributor",
"legacy source", "re-emerging concern", "poorly quantified source", "unresolved source",
"unaddressed challenge", "data gap",
"radiological exposure", "environmental contamination", "pathway human exposure", "vector environmental exposure",
"population-level exposure", "chronic exposure", "inhalation pathways", "ingestion pathways",
"environmental exposure dynamics", "long-term exposure", "indirect exposure", "cumulative exposure",
"mediate relationship", "modulate levels", "govern distribution", "control fate transport",
"mechanism", "facilitate mobilization", "conduit", "reservoir", "sink source", "cycling",
"regulate flux", "integral pathway", "oxidative stress", "inflammatory responses", "immune pathways",
"elevate risk", "amplify hazards", "exacerbate problem", "compound burdens", "risk multiplier",
"increase vulnerability", "threat multiplier", "heighten susceptibility", "intensify impacts", "worsen outcomes",
"adjustment", "independent confounders", "robust sensitivity", "persisted controlling",
"long-term exposure", "follow-up", "prospective association", "incident cases", "cumulative exposure",
"compared unexposed", "relative controls", "versus baseline",
"odds ratio", "hazard ratio", "relative risk", "standardized incidence ratio", "risk ratio",
"excess risk", "attributable risk", "positive correlation", "positive association",
"statistically significant", "confidence interval", "dose-response", "exposure-response"
]

NEGATIVE = ["not associated", "no association", "unrelated", "no significant"]




In [ ]:

import ast
df['tags'] = df['tags'].apply(ast.literal_eval)
all_tags = [tag for tags_list in df['tags'] for tag in tags_list]
unique_tags = set(all_tags)

print("Unique tags in your dataset:", unique_tags)


Unique tags in your dataset: {'B-heart_disease', 'I-diabetes', 'I-radiation_exposures', 'B-diabetes', 'I-chemical_exposures', 'B-hazardous_substances', 'B-contaminants', 'B-air_pollution', 'I-contaminants', 'I-other_FACTORS', 'B-occupational_exposures', 'I-hazardous_substances', 'B-built_environment', 'B-cancer', 'I-lung_disease', 'B-lung_disease', 'B-other_FACTORS', 'B-chemical_exposures', 'O', 'I-heart_disease', 'B-radiation_exposures', 'I-occupational_exposures', 'I-cancer', 'I-air_pollution'}


In [ ]:
import ast

def safe_literal_eval(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

df['tokens'] = df['tokens'].apply(safe_literal_eval)
df['tags'] = df['tags'].apply(safe_literal_eval)


In [ ]:
DISEASE_TAGS = ['B-cancer','I-cancer','B-lung_disease','I-lung_disease','B-heart_disease','I-heart_disease','B-diabetes','I-diabetes']
ENV_TAGS = ['B-chemical_exposures','I-chemical_exposures','B-radiation_exposures','I-radiation_exposures','B-occupational_exposures','I-occupational_exposures','B-air_pollution','I-air_pollution','B-climate_factors','I-climate_factors','B-built_environment','I-built_environment','B-hazardous_substances','I-hazardous_substances','B-contaminants','I-contaminants','B-other_FACTORS','I-other_FACTORS']


In [ ]:
import pandas as pd

relations = []

for _, row in df.iterrows():

    tokens = row["tokens"]
    tags = row["tags"]

    if not isinstance(tokens, list) or not isinstance(tags, list):
        continue
    if len(tokens) != len(tags):
        continue

    entities = []
    current_tokens = []
    current_tag = None

    for token, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if current_tokens:
                entities.append((current_tag, " ".join(current_tokens)))
            current_tokens = [token]
            current_tag = tag

        elif tag.startswith("I-") and current_tokens:
            current_tokens.append(token)

        else:
            if current_tokens:
                entities.append((current_tag, " ".join(current_tokens)))
                current_tokens = []
                current_tag = None

    if current_tokens:
        entities.append((current_tag, " ".join(current_tokens)))

    diseases = [text for tag, text in entities if tag in DISEASE_TAGS]
    factors = [text for tag, text in entities if tag in ENV_TAGS]


    if not diseases or not factors:
        continue


    sentence_lower = row["sentence"].lower()

    relation = "unknown"
    if any(k in sentence_lower for k in POSITIVE):
        relation = "positive"
    elif any(k in sentence_lower for k in NEGATIVE):
        relation = "negative"

    relations.append({


        "PMID": row.get("PMID", ""),
        "authors": row.get("authors", ""),
        "affiliation": row.get("affiliation", ""),
        "country": row.get("country", ""),
        "journal": row.get("journal", ""),
        "year": row.get("year", ""),
        "title": row.get("title", ""),
        "abstract": row.get("abstract", ""),
        "filtered_abstract": row.get("filtered_abstract", ""),
        "sentence": row['sentence'],
        "tokens": tokens,
        "tags": tags,
         "diseases": ", ".join(set(diseases)),
        "environmental_factors": ", ".join(set(factors)),
        "relation": relation
    })


relations_df = pd.DataFrame(relations)

print("Number of relations found:", len(relations_df))
relations_df.head()


Number of relations found: 10854


,PMID,authors,affiliation,country,journal,year,title,abstract,filtered_abstract,sentence,tokens,tags,diseases,environmental_factors,relation
0,41120039,He William J; Prescott Brenton R; Xanthakis Va...,"Framingham Heart Study, National Heart, Lung, ...",Usa,American heart journal,2026,Association of obesity subphenotypes with indi...,Previous studies have reported that obesity-re...,previous studies have reported that obesityrel...,previous studies have reported that obesityrel...,"[previous, studies, have, reported, that, obes...","[O, O, O, O, O, O, O, O, O, B-diabetes, O, O, ...","diabetes, cardiovascular disease, heart",lead,positive
1,41120010,Kaali Seyram; Li Michelle; Mujtaba Mohamed Nuh...,"Kintampo Health Research Centre, Research and ...",United States,Environmental research,2026,Household air pollution exposures over pregnan...,Household air pollution is a major contributor...,household air pollution is a major contributor...,household air pollution is a major contributor...,"[household, air, pollution, is, a, major, cont...","[O, B-hazardous_substances, I-hazardous_substa...",cardiovascular disease,air pollution,positive
2,40730107,Lee Haejung; Lee DaeEun; Jun Sukhyun,College of Nursing/Research Institute of Nursi...,Usa,Biological research for nursing,2026,Lifestyle and Genetic Factors Associated With ...,<b>Objective:</b> To identify single nucleotid...,bobjectiveb to identify single nucleotide poly...,bresultsb three snps rs8086325 rs34233878 rs21...,"[bresultsb, three, snps, rs8086325, rs34233878...","[O, O, O, O, O, O, O, O, O, O, B-heart_disease...",cvd,lead,positive
3,41297583,Cisneros-Mejorado Abraham; Colom-Casasnovas An...,"Instituto de Neurobiología, Universidad Nacion...",Spain,Neuropharmacology,2026,Purinergic signaling in Stroke and Multiple Sc...,Stroke and multiple sclerosis involve oxygen a...,stroke and multiple sclerosis involve oxygen a...,stroke and multiple sclerosis involve oxygen a...,"[stroke, and, multiple, sclerosis, involve, ox...","[B-heart_disease, O, O, O, O, O, O, O, O, O, O...",stroke,lead,positive
4,41241427,Mosnaim Giselle; Sweis Auddie M; Konsur Evelyn...,"Division of Allergy and Immunology, Department...",Usa,Immunology and allergy clinics of North America,2026,Differential Diagnoses Associated with Chronic...,Chronic rhinitis and chronic rhinosinusitis ca...,chronic rhinitis and chronic rhinosinusitis ca...,chronic rhinitis and chronic rhinosinusitis ca...,"[chronic, rhinitis, and, chronic, rhinosinusit...","[O, O, O, O, O, O, O, O, O, O, O, O, B-hazardo...","myocardial infarction, neoplasms, asthma, stroke",lead,positive


In [ ]:
relations_df = relations_df[relations_df['filtered_abstract'].notna()]
relations_df = relations_df[relations_df['filtered_abstract'].str.strip() != ""]

relations_df = relations_df.drop_duplicates(subset=["filtered_abstract"]).reset_index(drop=True)

print("After removing duplicates and nulls:", relations_df.shape)


After removing duplicates and nulls: (10197, 15)


In [ ]:

relations_df.to_excel("relations_lung.xlsx", index=False)
print("Dataset saved as cleaned_relations_heart.xlsx")


Dataset saved as cleaned_relations_heart.xlsx


In [ ]:
from google.colab import files

files.download("relations_lung.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>